In [ ]:
import json
from pathlib import Path

with open(Path("../data/site_removed_complexes.json"), "r") as f:
    ligand_interactions = json.load(f)



In [ ]:
from pathlib import Path
import subprocess
from tqdm import tqdm
from joblib import Parallel, delayed
import tempfile
import shutil

def process_pdb_file(pdb_file_path, smiles_output_dir, sdf_output_dir):
    with pdb_file_path.open('r') as file:
        lines = file.readlines()
    
    hetatm_blocks = {}
    ligands_found = 0
    
    for line in lines:
        if line.startswith("HETATM"):
            res_name = line[17:20].strip()
            chain_id = line[21].strip()
            key = f"{res_name}_{chain_id}"
            if key not in hetatm_blocks:
                hetatm_blocks[key] = []
            hetatm_blocks[key].append(line)
        elif line.startswith("END"):
            break
    
    if not hetatm_blocks:
        return 0
    
    ligands_found = len(hetatm_blocks)
    
    with tempfile.TemporaryDirectory() as hetatm_dir:
        hetatm_dir = Path(hetatm_dir)
        for key, hetatm_lines in hetatm_blocks.items():
            output_file = hetatm_dir / f"{key}.pdb"
            with output_file.open('w') as out_file:
                out_file.writelines(hetatm_lines)
        
        smiles_file = smiles_output_dir / pdb_file_path.with_suffix('.smi').name
        sdf_file = sdf_output_dir / pdb_file_path.with_suffix('.sdf').name
        
        with smiles_file.open('w') as out_smiles:
            for pdb_file in hetatm_dir.glob("*.pdb"):
                temp_smiles_file = pdb_file.with_suffix('.smi')
                temp_sdf_file = pdb_file.with_suffix('.sdf')
                
                command_smiles = ["obabel", str(pdb_file), "-O", str(temp_smiles_file), "-osmi"]
                subprocess.run(command_smiles, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
                
                command_sdf = ["obabel", str(pdb_file), "-O", str(temp_sdf_file), "-osdf"]
                subprocess.run(command_sdf, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
                
                if temp_smiles_file.exists():
                    with temp_smiles_file.open('r') as temp_file:
                        for line in temp_file:
                            smiles_line = f"{line.strip()}\t{pdb_file.stem}\n"
                            out_smiles.write(smiles_line)
                
                if temp_sdf_file.exists():
                    with sdf_file.open('a') as out_sdf:
                        with temp_sdf_file.open('r') as temp_file:
                            out_sdf.write(temp_file.read())
    
    return ligands_found

def convert_pdb_folder(input_dir, output_dir):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    smiles_output_dir = output_dir / "smi_files"
    sdf_output_dir = output_dir / "sdf_files"
    
    smiles_output_dir.mkdir(parents=True, exist_ok=True)
    sdf_output_dir.mkdir(parents=True, exist_ok=True)
    
    pdb_files = list(input_dir.glob("*.pdb"))
    
    results = Parallel(n_jobs=56)(
        delayed(process_pdb_file)(pdb_file, smiles_output_dir, sdf_output_dir)
        for pdb_file in tqdm(pdb_files, desc="Processing PDB files")
    )
    
    total_proteins = len(pdb_files)
    total_ligands = sum(results)
    
    print(f"Total proteins processed: {total_proteins}")
    print(f"Total ligands found: {total_ligands}")


convert_pdb_folder('/mnt/ligandpro/db/PDB/pdb2/processed/', './output_files')

Processing PDB files:  21%|██        | 45520/218235 [03:00<11:44, 245.09it/s]

In [ ]:
from Bio.PDB import PDBParser, NeighborSearch
import numpy as np

def calculate_center_of_mass(atoms):
    coords = np.array([atom.coord for atom in atoms])
    center_of_mass = np.mean(coords, axis=0)
    return center_of_mass

def find_pockets_and_centers(structure, radius=6.0):
    atoms = list(structure.get_atoms())
    ns = NeighborSearch(atoms)
    
    pockets = []
    
    for atom in atoms:
        neighbors = ns.search(atom.coord, radius, level='A')
        pocket_atoms = [n for n in neighbors if n.element != 'H']
        
        if len(pocket_atoms) > 0:
            center_of_mass = calculate_center_of_mass(pocket_atoms)
            chain_ids = set([atom.get_parent().get_parent().id for atom in pocket_atoms])
            pockets.append((center_of_mass, chain_ids))
    
    return pockets

def analyze_ligand_protein_interactions(file_path, radius=6.0):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('pdb_structure', file_path)
    
    pockets = find_pockets_and_centers(structure, radius=radius)
    
    single_chain_interacting_ligands = []
    interface_interacting_ligands = []
    
    for model in structure:
        for chain in model:
            for residue in chain:
                if residue.id[0] != ' ':
                    ligand_center = calculate_center_of_mass(list(residue.get_atoms()))
                    for pocket_center, chain_ids in pockets:
                        distance = np.linalg.norm(ligand_center - pocket_center)
                        if distance <= radius:
                            if len(chain_ids) == 1:
                                single_chain_interacting_ligands.append((residue.get_resname(), chain.id, distance))
                            else:
                                interface_interacting_ligands.append((residue.get_resname(), chain.id, distance, chain_ids))
                            break
    
    return single_chain_interacting_ligands, interface_interacting_ligands

file_path = '/mnt/ligandpro/db/PDB/pdb2/processed/pdb8p3w.pdb'
single_chain_ligands, interface_ligands = analyze_ligand_protein_interactions(file_path)

# Вывод результатов
print("Summary of ligand interactions:")
print(f"Total ligands interacting with single chain pockets: {len(single_chain_ligands)}")
print(f"Total ligands interacting with interface pockets (between chains): {len(interface_ligands)}\n")

print("Ligands interacting with single chain pockets:")
for ligand, chain_id, distance in single_chain_ligands:
    print(f"Ligand {ligand} in chain {chain_id} interacts with a single chain pocket within {distance:.2f} Å")

print("\nLigands interacting with interface pockets:")
for ligand, chain_id, distance, interacting_chains in interface_ligands:
    chains = ', '.join(interacting_chains)
    print(f"Ligand {ligand} in chain {chain_id} interacts with a pocket between chains {chains} within {distance:.2f} Å")


In [ ]:
from Bio.PDB import PDBIO, Select
from Bio.PDB.Polypeptide import PPBuilder
from Bio.Align import PairwiseAligner
from collections import defaultdict

class LigandSelect(Select):
    def __init__(self, ligand_resname, chain_id, interacting_chains):
        self.ligand_resname = ligand_resname
        self.chain_id = chain_id
        self.interacting_chains = interacting_chains

    def accept_chain(self, chain):
        return chain.id == self.chain_id or chain.id in self.interacting_chains

    def accept_residue(self, residue):
        if residue.get_resname() == self.ligand_resname and residue.get_parent().id == self.chain_id:
            return True
        elif residue.id[0] == ' ':
            return True
        else:
            return False

def extract_chain_sequence(structure, chain_id):
    ppb = PPBuilder()
    for chain in structure.get_chains():
        if chain.id == chain_id:
            polypeptides = ppb.build_peptides(chain)
            sequences = [str(pp.get_sequence()) for pp in polypeptides]
            return ''.join(sequences)
    return None

def save_ligand_structure_by_seq(structure, ligand_info, output_dir, saved_structures, original_lines):
    ligand_resname, chain_id, distance, interacting_chains = ligand_info
    io = PDBIO()
    io.set_structure(structure)
    
    main_chain_seq = extract_chain_sequence(structure, chain_id)
    aligner = PairwiseAligner()
    similar_seq_found = False

    if ligand_resname not in saved_structures:
        saved_structures[ligand_resname] = {}

    for saved_seq in saved_structures[ligand_resname]:
        alignment_score = aligner.align(main_chain_seq, saved_seq).score
        max_possible_score = max(len(main_chain_seq), len(saved_seq))
        similarity = alignment_score / max_possible_score

        if similarity >= 0.9:
            similar_seq_found = True
            if len(main_chain_seq) > len(saved_seq):
                saved_structures[ligand_resname].pop(saved_seq)
                break
            else:
                return

    if not similar_seq_found or len(main_chain_seq) > len(saved_seq):
        select = LigandSelect(ligand_resname, chain_id, interacting_chains)
        chains_str = "_".join(interacting_chains) if interacting_chains else chain_id
        output_file = str(Path(output_dir) / f"{ligand_resname}_chain_{chain_id}_interacts_with_{chains_str}.pdb")
        
        with open(output_file, 'w') as f_out:
            f_out.writelines(original_lines)
            io.save(f_out, select=select)
        
        saved_structures[ligand_resname][main_chain_seq] = structure

def load_original_lines(file_path):
    with open(file_path, 'r') as f:
        lines = [line for line in f if not line.startswith(("ATOM", "HETATM"))]
    return lines

output_dir = Path("separated_complexes_seq")
output_dir.mkdir(exist_ok=True)

original_lines = load_original_lines(file_path)

single_chain_ligands, interface_ligands = analyze_ligand_protein_interactions(file_path)

parser = PDBParser(QUIET=True)
structure = parser.get_structure('pdb_structure', file_path)

saved_structures = {}

for ligand_info in single_chain_ligands:
    ligand_resname, chain_id, distance = ligand_info
    save_ligand_structure_by_seq(structure, (ligand_resname, chain_id, distance, []), output_dir, saved_structures, original_lines)

for ligand_info in interface_ligands:
    save_ligand_structure_by_seq(structure, ligand_info, output_dir, saved_structures, original_lines)


In [ ]:
from Bio.PDB import PDBIO, Select
import MDAnalysis as mda
from MDAnalysis.analysis import align, rms
import numpy as np
from pathlib import Path
import tempfile

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="MDAnalysis.core.universe")


class LigandSelect(Select):
    def __init__(self, ligand_resname, chain_id, interacting_chains):
        self.ligand_resname = ligand_resname
        self.chain_id = chain_id
        self.interacting_chains = interacting_chains

    def accept_chain(self, chain):
        return chain.id == self.chain_id or chain.id in self.interacting_chains

    def accept_residue(self, residue):
        if residue.get_resname() == self.ligand_resname and residue.get_parent().id == self.chain_id:
            return True
        elif residue.id[0] == ' ':
            return True
        else:
            return False

def calculate_aligned_rmsd(structure1_file, structure2_file, chain_id):
    u1 = mda.Universe(structure1_file)
    u2 = mda.Universe(structure2_file)
    
    selection = f"chainID {chain_id} and backbone"
    
    atoms1 = u1.select_atoms(selection)
    atoms2 = u2.select_atoms(selection)
    
    aligner = align.AlignTraj(u2, u1, select=selection, in_memory=True)
    aligner.run()

    rmsd_value = rms.rmsd(atoms2.positions, atoms1.positions)
    
    return rmsd_value

def save_structure_to_tempfile(structure):
    io = PDBIO()
    temp_file = tempfile.NamedTemporaryFile(suffix=".pdb", delete=False)
    io.set_structure(structure)
    io.save(temp_file.name)
    temp_file.close()
    return temp_file.name

def save_ligand_structure_by_rmsd(structure, ligand_info, output_dir, saved_structures, original_lines, rmsd_threshold=2.0):
    ligand_resname, chain_id, distance, interacting_chains = ligand_info
    io = PDBIO()
    io.set_structure(structure)

    new_structure_file = save_structure_to_tempfile(structure)
    similar_structure_found = False

    if ligand_resname not in saved_structures:
        saved_structures[ligand_resname] = []

    for saved_structure_file in saved_structures[ligand_resname]:
        rmsd_value = calculate_aligned_rmsd(saved_structure_file, new_structure_file, chain_id)
        #print(f"Comparing {ligand_resname}_chain_{chain_id} with saved structure: RMSD = {rmsd_value:.3f}")
        if rmsd_value < rmsd_threshold:
            similar_structure_found = True
            break

    if not similar_structure_found:
        select = LigandSelect(ligand_resname, chain_id, interacting_chains)
        chains_str = "_".join(interacting_chains) if interacting_chains else chain_id
        output_file = str(Path(output_dir) / f"{ligand_resname}_chain_{chain_id}_interacts_with_{chains_str}.pdb")
        
        with open(output_file, 'w') as f_out:
            f_out.writelines(original_lines)
            io.save(f_out, select=select)
        
        saved_structures[ligand_resname].append(new_structure_file)
        #print(f"Saved structure {output_file}")

def load_original_lines(file_path):
    with open(file_path, 'r') as f:
        lines = [line for line in f if not line.startswith(("ATOM", "HETATM"))]
    return lines

output_dir = Path("separated_complexes_rmsd")
output_dir.mkdir(exist_ok=True)

original_lines = load_original_lines(file_path)

single_chain_ligands, interface_ligands = analyze_ligand_protein_interactions(file_path)

parser = PDBParser(QUIET=True)
structure = parser.get_structure('pdb_structure', file_path)

saved_structures = {}

for ligand_info in single_chain_ligands:
    ligand_resname, chain_id, distance = ligand_info
    save_ligand_structure_by_rmsd(structure, (ligand_resname, chain_id, distance, []), output_dir, saved_structures, original_lines)

for ligand_info in interface_ligands:
    save_ligand_structure_by_rmsd(structure, ligand_info, output_dir, saved_structures, original_lines)


здоров теперь на основе этого напиши мне два отдельных скрипта которые будут брать все файлы из PROCESSED_DIR = os.path.join(BASE_DIR, "pdb2/processed") 
BASE_DIR = "/mnt/ligandpro/db/PDB" и разбивать и сохранять разделённые структуры в новый путь pdb2/(какие то названия) надо принтом вывести сколько обработал структур каждый скрипт сколько нашёл обычных сколько на интерфейсе затем какой процент тех и тех и ещё дополнительной статистики (вывод оформи красиво) обработать надо все структур из пути с исопльзваетим tqdm и joblib пиши с доктрингами и без комментариев (важно что бы устойчиво к ошибкам в конкретных структурах (ошибки потом тоже вывести))

separated_complexes_seq
separated_complexes_rmsd